# Reddit API scrape: Wales communities

This notebook runs the `pushshiftreader scrape-reddit` command for a small test scrape of:

- `r/wales`
- `r/Cardiff`
- `r/southwales`
- `r/northwales`

Credentials are entered with hidden prompts and passed only to the subprocess environment. Do not write secrets directly into notebook cells.

## What this downloads

For each target subreddit, the scraper downloads recent submissions from the selected listing, plus comments for each downloaded submission. It writes JSONL files under `runs/reddit-api/...`:

```text
metadata.json
wales/submissions.jsonl
wales/comments.jsonl
Cardiff/submissions.jsonl
Cardiff/comments.jsonl
southwales/submissions.jsonl
southwales/comments.jsonl
northwales/submissions.jsonl
northwales/comments.jsonl
```

It does not download images, videos, private content, deleted content, full user histories, or historical archive data. Very large threads may not include every hidden `more` comment placeholder.

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/Users/alexjoegoddard/Documents/Studio/projects/pushshiftreader")

CLI = PROJECT_ROOT / ".venv" / "bin" / "pushshiftreader"
OUTPUT_ROOT = PROJECT_ROOT / "runs" / "reddit-api" / "wales-test"

print(f"Project root: {PROJECT_ROOT}")
print(f"CLI: {CLI}")
print(f"Output root: {OUTPUT_ROOT}")

In [ ]:
CREDENTIALS_PATH = PROJECT_ROOT / ".env.reddit"
EXAMPLE_CREDENTIALS_PATH = PROJECT_ROOT / ".env.reddit.example"

if not CREDENTIALS_PATH.exists():
    template = EXAMPLE_CREDENTIALS_PATH.read_text(encoding="utf-8")
    CREDENTIALS_PATH.write_text(template, encoding="utf-8")
    raise FileNotFoundError(
        f"Created {CREDENTIALS_PATH}. Open that file in VS Code, replace the placeholder values, save it, then rerun this cell."
    )

def read_env_file(path):
    values = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip("'").strip('"')
    return values

credentials = read_env_file(CREDENTIALS_PATH)
auth_mode = credentials.get("REDDIT_AUTH_MODE", "client_credentials").strip() or "client_credentials"
if auth_mode not in {"client_credentials", "installed_client"}:
    raise ValueError("REDDIT_AUTH_MODE must be client_credentials or installed_client")

required = ["REDDIT_CLIENT_ID", "REDDIT_USER_AGENT", "REDDIT_AUTH_MODE"]
if auth_mode == "client_credentials":
    required.append("REDDIT_CLIENT_SECRET")

credentials["REDDIT_AUTH_MODE"] = auth_mode
missing = [key for key in required if not credentials.get(key) or credentials[key].startswith("your_")]
if missing:
    raise ValueError(f"Fill in these values in {CREDENTIALS_PATH}: {', '.join(missing)}")

env = os.environ.copy()
env.update({key: credentials.get(key, "") for key in ["REDDIT_CLIENT_ID", "REDDIT_CLIENT_SECRET", "REDDIT_USER_AGENT", "REDDIT_AUTH_MODE", "REDDIT_DEVICE_ID"]})

print(f"Loaded Reddit credentials from {CREDENTIALS_PATH} into this notebook session.")

## First small test

This first run is intentionally small: 10 submissions per subreddit from the past 30 days, plus up to 100 comments per submission. That is enough to verify credentials, output layout, and rate-limit behavior without pulling a large sample.

In [ ]:
cmd = [
    str(CLI),
    "scrape-reddit",
    "--output", str(OUTPUT_ROOT),
    "--sort", "new",
    "--days", "30",
    "--max-submissions", "10",
    "--comments-limit", "100",
]

print("Running:", " ".join(cmd[:4]), "...")
result = subprocess.run(
    cmd,
    cwd=PROJECT_ROOT,
    env=env,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)
    raise RuntimeError(f"Scrape failed with exit code {result.returncode}")

In [ ]:
metadata_path = OUTPUT_ROOT / "metadata.json"
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

print(json.dumps({
    "subreddits": metadata["subreddits"],
    "total_submissions": metadata["total_submissions"],
    "total_comments": metadata["total_comments"],
    "output": str(OUTPUT_ROOT),
}, indent=2))

In [ ]:
# Preview the first submission record without printing all downloaded text.
for subreddit in metadata["subreddits"]:
    path = OUTPUT_ROOT / subreddit / "submissions.jsonl"
    if path.exists() and path.stat().st_size:
        first = json.loads(path.read_text(encoding="utf-8").splitlines()[0])
        print({
            "subreddit": first.get("subreddit"),
            "id": first.get("id"),
            "created_utc": first.get("created_utc"),
            "title": first.get("title"),
            "num_comments": first.get("num_comments"),
            "full_link": first.get("full_link"),
        })
        break

## Spreadsheet exports

The live API scraper writes JSONL, not the Pushshift Parquet layout. Run this cell to create flat CSV files that spreadsheet software can open.

In [ ]:
def iter_jsonl(path):
    if not path.exists():
        return
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)

def write_csv(records, output_path, fields):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        for record in records:
            writer.writerow(record)

submission_fields = [
    "subreddit", "id", "name", "author", "created_utc", "title", "selftext",
    "url", "permalink", "full_link", "score", "upvote_ratio", "num_comments",
    "over_18", "spoiler", "stickied", "locked", "is_self", "link_flair_text", "retrieved_at",
]
comment_fields = [
    "subreddit", "submission_id", "id", "name", "author", "created_utc", "body",
    "link_id", "parent_id", "score", "depth", "is_submitter", "stickied",
    "distinguished", "permalink", "full_link", "retrieved_at",
]

all_submissions = []
all_comments = []
for subreddit in metadata["subreddits"]:
    all_submissions.extend(iter_jsonl(OUTPUT_ROOT / subreddit / "submissions.jsonl") or [])
    all_comments.extend(iter_jsonl(OUTPUT_ROOT / subreddit / "comments.jsonl") or [])

submissions_csv = OUTPUT_ROOT / "all_submissions.csv"
comments_csv = OUTPUT_ROOT / "all_comments.csv"
write_csv(all_submissions, submissions_csv, submission_fields)
write_csv(all_comments, comments_csv, comment_fields)

print(f"Wrote {len(all_submissions)} submissions to {submissions_csv}")
print(f"Wrote {len(all_comments)} comments to {comments_csv}")

## Larger run

After the small test succeeds, increase the limits. The scraper watches Reddit's `x-ratelimit-*` response headers and sleeps when the remaining request budget is low.

For a broad first pass, use `--skip-comments` and a high submission limit. That keeps the run to listing-page requests only. `--days 30` keeps this to submissions from the past month.

In [ ]:
# Uncomment to run a larger scrape after checking the test output.
# OUTPUT_ROOT = PROJECT_ROOT / "runs" / "reddit-api" / "wales-submissions-30d"
# cmd = [
#     str(CLI),
#     "scrape-reddit",
#     "--output", str(OUTPUT_ROOT),
#     "--sort", "new",
#     "--days", "30",
#     "--max-submissions", "1000",
#     "--skip-comments",
# ]
# result = subprocess.run(cmd, cwd=PROJECT_ROOT, env=env, text=True, capture_output=True)
# print(result.stdout)
# if result.returncode != 0:
#     print(result.stderr, file=sys.stderr)
#     raise RuntimeError(f"Scrape failed with exit code {result.returncode}")